# Chunking MinerU Markdown

Splits MinerU output (`markdown.md` + `structured_content.json`) into text, table, formula, code and image chunks.
Run [document_convert2md_example.ipynb](document_convert2md_example.ipynb) first. Requires `chonkie`; the semantic
chunker also needs `sentence-transformers`.

In [1]:
from collections import Counter
from pathlib import Path

from ragu.chunker.markdown_chunkers import RecursiveMarkdownChunker, SemanticMarkdownChunker
from ragu.chunker.types import Document

documents = [Document.from_mineru(path.parent) for path in sorted(Path("data/md/result").rglob("markdown.md"))]
print("\n".join([document.title for document in documents]))

c:\Dev\RAGU\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Lithium-Ion Batteries: Chemistry, Materials and Performance
RAGU: A Multi-Step GraphRAG Engine with a Compact Domain-Adapted LLM
Silicon Solar Cells: Principles and Performance
The Transistor: History, Physics and Materials


## Recursive chunker

In [2]:
chunks = RecursiveMarkdownChunker(max_chunk_size=1000).split(documents)
print(Counter(type(chunk).__name__ for chunk in chunks))

Counter({'TextChunk': 80, 'TableChunk': 13, 'FormulaChunk': 11, 'ImageChunk': 6, 'CodeChunk': 1})


In [5]:
for name in ("TextChunk", "TableChunk", "FormulaChunk", "CodeChunk", "ImageChunk"):
    chunk = next((chunk for chunk in chunks if type(chunk).__name__ == name), None)
    if chunk:
        print(f"[{name}] {chunk.metadata}")
        print(f"--- to_embed\n{chunk.to_embed()[:300]}")
        print(f"--- to_llm\n{chunk.to_llm()[:500]}\n")
        print("=" * 100 + "\n")

[TextChunk] {'document_title': 'Lithium-Ion Batteries: Chemistry, Materials and Performance', 'section': 'Lithium-Ion Batteries: Chemistry, Materials and Performance'}
--- to_embed
Lithium-Ion Batteries: Chemistry, Materials and Performance

*A short technical overview prepared as a sample document for document parsing.*
--- to_llm
Document: Lithium-Ion Batteries: Chemistry, Materials and Performance
Section: Lithium-Ion Batteries: Chemistry, Materials and Performance

*A short technical overview prepared as a sample document for document parsing.*


[TableChunk] {'document_title': 'Lithium-Ion Batteries: Chemistry, Materials and Performance', 'section': '1. History', 'description': 'Year Event Organisation People 1976 Lithium metal / TiS₂ rechargeable cell Exxon M. S. Whittingham 1980 LiCoO₂ cathode proposed University of Oxford J. B. Goodenough 1985 Carbon anode, first practical Li-ion prototype Asahi Kasei A. Yoshino 1991 First commercial Li-ion cell Sony — 1997 LiFePO₄ (olivine) ca

All chunks from parse result

In [6]:
for chunk in chunks:
    if chunk:
        print(f"[{chunk.__class__.__name__}] {chunk.metadata}")
        print(f"--- to_embed\n{chunk.to_embed()[:200]}")
        print(f"--- to_llm\n{chunk.to_llm()[:200]}\n...TRUNCETED...\n")
        print("=" * 100 + "\n")

[TextChunk] {'document_title': 'Lithium-Ion Batteries: Chemistry, Materials and Performance', 'section': 'Lithium-Ion Batteries: Chemistry, Materials and Performance'}
--- to_embed
Lithium-Ion Batteries: Chemistry, Materials and Performance

*A short technical overview prepared as a sample document for document parsing.*
--- to_llm
Document: Lithium-Ion Batteries: Chemistry, Materials and Performance
Section: Lithium-Ion Batteries: Chemistry, Materials and Performance

*A short technical overview prepared as a sample document fo
...TRUNCETED...


[TextChunk] {'document_title': 'Lithium-Ion Batteries: Chemistry, Materials and Performance', 'section': '1. History'}
--- to_embed
1. History

The first rechargeable lithium cell was built in the 1970s at **Exxon** by **M. Stanley Whittingham**, who used a titanium disulfide cathode. In 1980 **John B. Goodenough** showed that lit
--- to_llm
Document: Lithium-Ion Batteries: Chemistry, Materials and Performance
Section: 1. History

The first re

## Semantic chunker

In [ ]:
semantic_chunks = SemanticMarkdownChunker(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", max_chunk_size=256, device="cpu"
).split(documents)
print(Counter(type(chunk).__name__ for chunk in semantic_chunks))
print(Counter(chunk.metadata.get("section") for chunk in semantic_chunks))

## Descriptions with an LLM

`a_split` also writes `metadata["description"]` of tables, formulas, code and images with the model passed as `llm`
(images are sent as pictures, so use a vision model for them). Needs `OPENAI_BASE_URL`, `OPENAI_API_KEY`,
`LLM_MODEL_NAME`.

In [ ]:
import os

from ragu.chunker.types import ObjectChunk
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI

client = CachedAsyncOpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key=os.environ["OPENAI_API_KEY"])
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])

described = await RecursiveMarkdownChunker(max_chunk_size=1000, llm=llm).a_split(documents)
for chunk in [chunk for chunk in described if isinstance(chunk, ObjectChunk)][:5]:
    print(f"[{type(chunk).__name__}] {chunk.metadata.get('caption')}\n{chunk.to_embed()}\n")